# Web Server Log Analysis - Python Take-Home Assessment

## Overview
This assessment involves analyzing the Calgary HTTP dataset, which contains approximately one year's worth of HTTP requests to the University of Calgary's Computer Science web server. You'll work with real-world web server log data to extract meaningful insights and demonstrate your Python data analysis skills.

## Part 1: Data Loading and Cleaning

### Instructions

* Work in the cells below - You can add as many cells as needed for data loading, cleaning, and exploration
* Import required libraries
* Implement data loading and cleaning - Create functions to download, parse, and clean the log data
* Explore the data - Understand the structure and identify any data quality issues

In [5]:
# You can write your code here for data loading, cleaning, and exploration. Add cells as necessary.

In [6]:
import gzip
import pandas as pd
from urllib.request import urlopen

url = "ftp://ita.ee.lbl.gov/traces/calgary_access_log.gz"

# Read ALL lines safely (no parsing)
with urlopen(url) as f:
    with gzip.GzipFile(fileobj=f) as gz:
        lines = [line.decode('latin1').strip() for line in gz]

# Create DataFrame (single column)
df = pd.DataFrame(lines, columns=["raw_log"])

print("Shape:", df.shape)
print(df.head())

Shape: (726739, 1)
                                             raw_log
0  local - - [24/Oct/1994:13:41:41 -0600] "GET in...
1  local - - [24/Oct/1994:13:41:41 -0600] "GET 1....
2  local - - [24/Oct/1994:13:43:13 -0600] "GET in...
3  local - - [24/Oct/1994:13:43:14 -0600] "GET 2....
4  local - - [24/Oct/1994:13:43:15 -0600] "GET 3....


In [7]:
import pandas as pd

# Show full column content (no ...)
pd.set_option('display.max_colwidth', None)

# Optional (better display)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)

In [8]:
print(df.head(1))

                                                                    raw_log
0  local - - [24/Oct/1994:13:41:41 -0600] "GET index.html HTTP/1.0" 200 150


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 726739 entries, 0 to 726738
Data columns (total 1 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   raw_log  726739 non-null  object
dtypes: object(1)
memory usage: 5.5+ MB


### Extract structured columns

In [11]:
import re

pattern = re.compile(
    r'(\S+)\s+\S+\s+\S+\s+\[(.*?)\]\s+"(.*?)"\s+(\d{3})\s+(\S+)'
)

df[['host', 'timestamp', 'request', 'status', 'size']] = df['raw_log'].str.extract(pattern)

In [12]:
df.head()

,raw_log,host,timestamp,request,status,size
0,"local - - [24/Oct/1994:13:41:41 -0600] ""GET index.html HTTP/1.0"" 200 150",local,24/Oct/1994:13:41:41 -0600,GET index.html HTTP/1.0,200,150
1,"local - - [24/Oct/1994:13:41:41 -0600] ""GET 1.gif HTTP/1.0"" 200 1210",local,24/Oct/1994:13:41:41 -0600,GET 1.gif HTTP/1.0,200,1210
2,"local - - [24/Oct/1994:13:43:13 -0600] ""GET index.html HTTP/1.0"" 200 3185",local,24/Oct/1994:13:43:13 -0600,GET index.html HTTP/1.0,200,3185
3,"local - - [24/Oct/1994:13:43:14 -0600] ""GET 2.gif HTTP/1.0"" 200 2555",local,24/Oct/1994:13:43:14 -0600,GET 2.gif HTTP/1.0,200,2555
4,"local - - [24/Oct/1994:13:43:15 -0600] ""GET 3.gif HTTP/1.0"" 200 36403",local,24/Oct/1994:13:43:15 -0600,GET 3.gif HTTP/1.0,200,36403


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 726739 entries, 0 to 726738
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   raw_log    726739 non-null  object
 1   host       724910 non-null  object
 2   timestamp  724910 non-null  object
 3   request    724910 non-null  object
 4   status     724910 non-null  object
 5   size       724910 non-null  object
dtypes: object(6)
memory usage: 33.3+ MB


Total rows: 726,739
Some columns have missing values (~1,800 rows)
All columns are currently of type object

### Inference 
* As you can see in info their is  wrong datatype  of datetime and size column 

### Flag invalid rows

In [17]:
df['is_valid'] = df['host'].notna()

print("Valid:", df['is_valid'].sum())
print("Invalid:", (~df['is_valid']).sum())

Valid: 724910
Invalid: 1829


### Parsing Timestamp to Datetime

In [19]:
# Remove timezone
df['timestamp'] = df['timestamp'].str.replace(r' -\d{4}', '', regex=True)

# Convert to datetime
df['timestamp'] = pd.to_datetime(
    df['timestamp'],
    format='%d/%b/%Y:%H:%M:%S',
    errors='coerce'
)

Inference
  * Timestamp converted from string → datetime format
- Enables:
  * Time-based analysis
  * Sorting & filtering
- Invalid timestamps become NaT (handled safely)

### Extract Request Components

In [22]:
split_cols = df['request'].str.split(' ', n=2, expand=True)

df['method'] = split_cols[0]
df['filename'] = split_cols[1]
df['protocol'] = split_cols[2]

In [23]:
df['method'] = df['method'].fillna('UNKNOWN')
df['filename'] = df['filename'].fillna('UNKNOWN')
df['protocol'] = df['protocol'].fillna('UNKNOWN')

In [24]:
print("Method UNKNOWN:", (df['method'] == 'UNKNOWN').sum())
print("Filename UNKNOWN:", (df['filename'] == 'UNKNOWN').sum())
print("Protocol UNKNOWN:", (df['protocol'] == 'UNKNOWN').sum())

Method UNKNOWN: 1829
Filename UNKNOWN: 1829
Protocol UNKNOWN: 2700


In [25]:
df[df['method'] == 'UNKNOWN'][['raw_log']]

,raw_log
128,local 780 index.html
129,local index.html
458,local index.html
5206,"remote - - [27/Oct/1994:18:51:50 -0600] ""GET index.html 200 3185"
5458,"remote - - [27/Oct/1994:23:17:17 -0600] ""GET index.html 200 3185"
...,...
723208,"remote - - [10/Oct/1995:16:59:18 -0600] ""GET index.html 404 -"
723435,local index.html
723436,local index.html
724086,local index.html


In [26]:
df.head(1)

,raw_log,host,timestamp,request,status,size,is_valid,method,filename,protocol
0,"local - - [24/Oct/1994:13:41:41 -0600] ""GET index.html HTTP/1.0"" 200 150",local,1994-10-24 13:41:41,GET index.html HTTP/1.0,200,150,True,GET,index.html,HTTP/1.0


### Extract File Extensions

In [28]:
df['file_extension'] = df['filename'].str.extract(r'\.(\w+)$')

In [29]:
df.head(1)

,raw_log,host,timestamp,request,status,size,is_valid,method,filename,protocol,file_extension
0,"local - - [24/Oct/1994:13:41:41 -0600] ""GET index.html HTTP/1.0"" 200 150",local,1994-10-24 13:41:41,GET index.html HTTP/1.0,200,150,True,GET,index.html,HTTP/1.0,html


- Inference
  * Extracts file types like:
    * html, gif, jpg

### Convert Data Types

In [32]:
df['status'] = pd.to_numeric(df['status'], errors='coerce')
df['size'] = pd.to_numeric(df['size'], errors='coerce')

- Inference
  * status → integer 
  * size → numeric 
  * Invalid values (like '-') become NaN

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 726739 entries, 0 to 726738
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   raw_log         726739 non-null  object        
 1   host            724910 non-null  object        
 2   timestamp       724910 non-null  datetime64[ns]
 3   request         724910 non-null  object        
 4   status          724910 non-null  float64       
 5   size            666804 non-null  float64       
 6   is_valid        726739 non-null  bool          
 7   method          726739 non-null  object        
 8   filename        726739 non-null  object        
 9   protocol        726739 non-null  object        
 10  file_extension  723410 non-null  object        
dtypes: bool(1), datetime64[ns](1), float64(2), object(7)
memory usage: 56.1+ MB


In [35]:
df.head(1)

,raw_log,host,timestamp,request,status,size,is_valid,method,filename,protocol,file_extension
0,"local - - [24/Oct/1994:13:41:41 -0600] ""GET index.html HTTP/1.0"" 200 150",local,1994-10-24 13:41:41,GET index.html HTTP/1.0,200.0,150.0,True,GET,index.html,HTTP/1.0,html


### Handling Missing Values

In [37]:
df.isnull().sum()

raw_log               0
host               1829
timestamp          1829
request            1829
status             1829
size              59935
is_valid              0
method                0
filename              0
protocol              0
file_extension     3329
dtype: int64

### 1. Handling Invalid / Malformed Row

In [39]:
df['is_valid'] = df['host'].notna()

# Clean dataset for analysis
df_clean = df[df['is_valid']].copy()

**Inference :** 

* **host, timestamp, request, status** These fields are critical for log interpretation. Missing values indicate malformed entries, so such rows are excluded from analytical processing while being retained in the original dataset for traceability.

#### Handling size Column

- Observation
   * ~59,935 missing values
   * Represented as - in logs

In [43]:
df_clean['size'] = pd.to_numeric(df_clean['size'], errors='coerce')
df_clean['size'] = df_clean['size'].fillna(0)

In [44]:
df_clean['size'].isna().sum()

np.int64(0)

Missing size values indicate no data transfer (e.g., errors like 404), hence replacing with 0 

#### Handling file_extension

In [47]:
df_clean['file_extension'] = df_clean['file_extension'].fillna('no_extension')

Inference

Some resources do not have extensions; labeling them as 'no_extension' preserves their analytical value.

# Final Clean Dataset

In [50]:
print(df_clean.isnull().sum())
print(df_clean.shape)
df_clean.head(3)

raw_log           0
host              0
timestamp         0
request           0
status            0
size              0
is_valid          0
method            0
filename          0
protocol          0
file_extension    0
dtype: int64
(724910, 11)


,raw_log,host,timestamp,request,status,size,is_valid,method,filename,protocol,file_extension
0,"local - - [24/Oct/1994:13:41:41 -0600] ""GET index.html HTTP/1.0"" 200 150",local,1994-10-24 13:41:41,GET index.html HTTP/1.0,200.0,150.0,True,GET,index.html,HTTP/1.0,html
1,"local - - [24/Oct/1994:13:41:41 -0600] ""GET 1.gif HTTP/1.0"" 200 1210",local,1994-10-24 13:41:41,GET 1.gif HTTP/1.0,200.0,1210.0,True,GET,1.gif,HTTP/1.0,gif
2,"local - - [24/Oct/1994:13:43:13 -0600] ""GET index.html HTTP/1.0"" 200 3185",local,1994-10-24 13:43:13,GET index.html HTTP/1.0,200.0,3185.0,True,GET,index.html,HTTP/1.0,html


## ⚠️ IMPORTANT: Template Questions Section
**DO NOT MODIFY THE TEMPLATE BELOW THIS POINT**

The following section contains the assessment questions. You may add cells above this section for data loading, cleaning, and exploration, but do not modify the function signatures or structure of the questions below.

## Part 2: Analysis Questions

### Instructions

* Implement each function according to its docstring specifications
* Use the cleaned data you prepared in Part 1
* Ensure your functions return the exact data types specified
* Test your functions to verify they work correctly
* You may add helper functions, but keep the main function signatures unchanged

### Q1: Count of total log records

In [55]:
def total_log_records() -> int:
    return len(df_clean)


answer1 = total_log_records()
print("Answer 1:")
print(answer1)

Answer 1:
724910


### Q2: Count of unique hosts

In [57]:
def unique_host_count() -> int:
    return df_clean['host'].nunique()


answer2 = unique_host_count()
print("Answer 2:")
print(answer2)

Answer 2:
2


### Q3: Date-wise unique filename counts

In [59]:
def datewise_unique_filename_counts() -> dict[str, int]:
    df_clean['date'] = df_clean['timestamp'].dt.strftime('%d-%b-%Y')
    
    return df_clean.groupby('date')['filename'].nunique().to_dict()

answer3 = datewise_unique_filename_counts()
print("Answer 3:")
print(answer3)

Answer 3:
{'01-Apr-1995': 438, '01-Aug-1995': 684, '01-Dec-1994': 271, '01-Feb-1995': 625, '01-Jan-1995': 88, '01-Jul-1995': 388, '01-Jun-1995': 591, '01-Mar-1995': 582, '01-May-1995': 467, '01-Nov-1994': 415, '01-Oct-1995': 554, '01-Sep-1995': 328, '02-Apr-1995': 466, '02-Aug-1995': 857, '02-Dec-1994': 325, '02-Feb-1995': 524, '02-Jan-1995': 141, '02-Jul-1995': 400, '02-Jun-1995': 515, '02-Mar-1995': 601, '02-May-1995': 702, '02-Nov-1994': 430, '02-Oct-1995': 873, '02-Sep-1995': 352, '03-Apr-1995': 795, '03-Aug-1995': 584, '03-Dec-1994': 189, '03-Feb-1995': 570, '03-Jan-1995': 311, '03-Jul-1995': 438, '03-Jun-1995': 401, '03-Mar-1995': 510, '03-May-1995': 609, '03-Nov-1994': 460, '03-Oct-1995': 848, '03-Sep-1995': 214, '04-Apr-1995': 821, '04-Aug-1995': 717, '04-Dec-1994': 213, '04-Feb-1995': 561, '04-Jan-1995': 333, '04-Jul-1995': 612, '04-Jun-1995': 371, '04-Mar-1995': 413, '04-May-1995': 684, '04-Nov-1994': 404, '04-Oct-1995': 893, '04-Sep-1995': 343, '05-Apr-1995': 891, '05-Aug-19

### Q4: Number of 404 response codes

In [61]:
def count_404_errors() -> int:
    return (df_clean['status'] == 404).sum()


answer4 = count_404_errors()
print("Answer 4:")
print(answer4)

Answer 4:
23586


### Q5: Top 15 filenames with 404 responses

In [63]:
def top_15_filenames_with_404() -> list[tuple[str, int]]:
    return list(
        df_clean[df_clean['status'] == 404]['filename']
        .value_counts()
        .head(15)
        .items()
    )


answer5 = top_15_filenames_with_404()
print("Answer 5:")
print(answer5)

Answer 5:
[('index.html', 4737), ('4115.html', 902), ('1611.html', 649), ('5698.xbm', 585), ('710.txt', 408), ('2002.html', 259), ('2177.gif', 193), ('10695.ps', 161), ('6555.html', 153), ('487.gif', 152), ('151.html', 149), ('488.gif', 148), ('3414.gif', 148), ('40.html', 148), ('9678.gif', 142)]


### Q6: Top 15 file extension with 404 responses

In [65]:
def top_15_ext_with_404() -> list[tuple[str, int]]:
    return list(
        df_clean[df_clean['status'] == 404]['file_extension']
        .value_counts()
        .head(15)
        .items()
    )


answer6 = top_15_ext_with_404()
print("Answer 6:")
print(answer6)

Answer 6:
[('html', 12213), ('gif', 7213), ('xbm', 826), ('ps', 754), ('no_extension', 563), ('jpg', 527), ('txt', 497), ('GIF', 135), ('htm', 109), ('cgi', 77), ('com', 46), ('Z', 41), ('dvi', 40), ('ca', 36), ('hmtl', 30)]


### Q7: Total bandwidth transferred per day for the month of July 1995

In [67]:
def total_bandwidth_per_day() -> dict[str, int]:
    july = df_clean[
        (df_clean['timestamp'].dt.month == 7) &
        (df_clean['timestamp'].dt.year == 1995)
    ]
    
    return (
        july.groupby(july['timestamp'].dt.strftime('%d-%b-%Y'))['size']
        .sum()
        .astype(int)
        .to_dict()
    )


answer7 = total_bandwidth_per_day()
print("Answer 7:")
print(answer7)

Answer 7:
{'01-Jul-1995': 11349799, '02-Jul-1995': 8656918, '03-Jul-1995': 13596612, '04-Jul-1995': 26573988, '05-Jul-1995': 19541225, '06-Jul-1995': 19755015, '07-Jul-1995': 9427822, '08-Jul-1995': 5403491, '09-Jul-1995': 4660556, '10-Jul-1995': 14917754, '11-Jul-1995': 22507207, '12-Jul-1995': 17367065, '13-Jul-1995': 15989234, '14-Jul-1995': 19186430, '15-Jul-1995': 15773233, '16-Jul-1995': 9016378, '17-Jul-1995': 19601338, '18-Jul-1995': 17099761, '19-Jul-1995': 17851725, '20-Jul-1995': 20752623, '21-Jul-1995': 25491617, '22-Jul-1995': 8136259, '23-Jul-1995': 9593870, '24-Jul-1995': 22308265, '25-Jul-1995': 24561635, '26-Jul-1995': 24995540, '27-Jul-1995': 25969995, '28-Jul-1995': 36460693, '29-Jul-1995': 11700624, '30-Jul-1995': 23189598, '31-Jul-1995': 30730715}


### Q8: Hourly request distribution

In [69]:
def hourly_request_distribution() -> dict[int, int]:
    return (
        df_clean.groupby(df_clean['timestamp'].dt.hour)
        .size()
        .to_dict()
    )


answer8 = hourly_request_distribution()
print("Answer 8:")
print(answer8)

Answer 8:
{0: 18764, 1: 14389, 2: 12694, 3: 10901, 4: 9981, 5: 10804, 6: 13059, 7: 16675, 8: 26591, 9: 33992, 10: 43377, 11: 47588, 12: 46818, 13: 51460, 14: 54563, 15: 50379, 16: 51181, 17: 45080, 18: 33222, 19: 30574, 20: 29698, 21: 27409, 22: 23828, 23: 21883}


### Q9: Top 10 most requested filenames

In [71]:
def top_10_most_requested_filenames() -> list[tuple[str, int]]:
    return list(
        df_clean['filename']
        .value_counts()
        .head(10)
        .items()
    )


answer9 = top_10_most_requested_filenames()
print("Answer 9:")
print(answer9)

Answer 9:
[('index.html', 140076), ('3.gif', 24006), ('2.gif', 23606), ('4.gif', 8018), ('244.gif', 5149), ('5.html', 5010), ('4097.gif', 4874), ('8870.jpg', 4493), ('6733.gif', 4278), ('8472.gif', 3843)]


### Q10: HTTP response code distribution

In [73]:
def response_code_distribution() -> dict[int, int]:
    return (
        df_clean['status']
        .value_counts()
        .astype(int)
        .to_dict()
    )


answer10 = response_code_distribution()
print("Answer 10:")
print(answer10)

Answer 10:
{200.0: 568348, 304.0: 97792, 302.0: 30295, 404.0: 23586, 403.0: 4743, 401.0: 46, 501.0: 43, 500.0: 42, 400.0: 15}
